shema: x, y, z, gene, tile_x, tile_y, tile_z, roi_id, slide_id

In [ ]:
def txt2parquet(sample_id, segmentation_mask, spots_path, output_dir):
    import pandas as pd
    import numpy as np
    import tifffile
    from pathlib import Path
    import glob
    
    mask_np = tifffile.imread(segmentation_mask) # loads full image into memory - not perfect, but works for this case

    print('loaded mask')
    mask_bool = mask_np > 0

    spots = pd.read_csv(spots_path, sep='\t', header=None, )
    # drop all columns except first 4
    spots = spots.iloc[:, :4]
    spots.columns = ['global_x', 'global_y', 'global_z', 'gene']
    spots = spots[spots['gene'] != 'Duplicated']

    print('loaded spots')

    spots['overlaps_nucleus'] = mask_bool[spots['global_y'].astype(int), spots['global_x'].astype(int)]
    spots['transcript_id'] = np.arange(len(spots))
    spots['cell_id'] = mask_np[spots['global_y'].astype(int), spots['global_x'].astype(int)]
    spots['cell_id'] = spots['cell_id'].replace(0, -1)
    spots['EntityID'] = spots['cell_id'].copy() # needed yes/no??
    spots.to_parquet(f'{output_dir}/{sample_id}/transcripts.parquet', index=False)
    print(f"Saved Parquet for sample {sample_id}")

Main script for running the function

In [ ]:
from pathlib import Path
import glob
import os

# --- CONFIGURATION: fill in paths before running ---
MC_OVERVIEW_CSV = Path("")       # path to cellposesam_segger_sample_overview.csv
INPUT_DIR = Path("")              # output directory (input_for_segger)
MC_SEGMENTATIONS_DIR = Path("")  # directory containing .tif segmentation masks
SDS_BASE_DIR = Path("")          # your SDS mount path (replaces "$HOME/sds-hd/" prefix in spot_table paths)

path_table = pd.read_csv(MC_OVERVIEW_CSV)
output_dir = str(INPUT_DIR)

for spots in path_table.spot_table:
    sample_ID = Path(spots).stem.split('_results')[0]
    beast_spots_path = spots.replace("$HOME/sds-hd/", str(SDS_BASE_DIR) + "/")

    #searching for Mask file
    pattern = os.path.join(str(MC_SEGMENTATIONS_DIR), f"{sample_ID}*.tif")
    mask = glob.glob(pattern)

    #print(f"Processing sample: {sample_ID}")
    txt2parquet(sample_ID, mask, beast_spots_path, output_dir)
